# squeeze_extreme_v4 — die-level stacking + SHAP X-stacking

v3와의 차이:
- **SHAP 통합**: `build_shap_features.py`가 만든 die-level SHAP 캐시(`shap_cache/<tag>/die_shap.npz`)를
  메타 입력에 추가. `shap_mode` 2종:
  * `always_include` — subset search는 base만, 메타 학습 시 SHAP 항상 포함 (v2 기본 동작 die-level화)
  * `searchable`     — base와 SHAP을 통합 풀로 두고 ridge/L1이 자동 가중치 학습
- v3의 `compet_xs_data.csv` 기반 raw X 핵심피처 경로(`build_die_matrix_with_extras`)는 **제거**됨 — 그 자리를 SHAP이 차지.
- **선행 조건**: SHAP 캐시는 v4 시점부터 `run_wf_xy` 키를 npz에 함께 저장해야 매칭 가능.
  구버전 캐시는 `run_shap_all.py --force`로 재생성 필요.

결과 저장: `4_output/04_stacking/results_extreme_v4/run_{ts}/` — die + unit CSV, summary JSON/CSV, best weights JSON (SHAP 컬럼 가중치 포함).


## 0. 환경 + setup


In [1]:
import os, sys

try:
    import google.colab
    if not os.path.exists("/content/project/setup.py"):
        os.system("pip install -q gdown")
        os.system("gdown --id 1AD4PDBnDVjp-LSna6puB7qLnpBqB7j_I -O /content/code.zip")
        os.system("unzip -qo /content/code.zip -d /content/project")
        os.makedirs("/content/project/0_data", exist_ok=True)
        os.system("gdown --id 1yOUo0_wPLcuZBSJIK592b00YkSIlk4zO -O /content/project/0_data/dataset.zip")
        os.system("unzip -qo /content/project/0_data/dataset.zip -d /content/project/0_data")
        os.remove("/content/project/0_data/dataset.zip")
    sys.path.insert(0, "/content/project")
    %run /content/project/setup.py
except ImportError:
    %run ../../setup.py

setup 완료


In [2]:
# _v4 패키지 import (이 노트북이 04_stacking/ 에 있으므로 같은 폴더의 _v4/ 가 보임)
import sys
from pathlib import Path
_HERE = Path.cwd() / "3_modeling" / "04_stacking" if not (Path.cwd() / "_v4").exists() else Path.cwd()
if str(_HERE) not in sys.path:
    sys.path.insert(0, str(_HERE))

from _v4.config import SqueezeV4Config
from _v4 import discovery, aggregate, search, weights, io
from _v4.records import score_rec

import numpy as np
import pandas as pd


## 1. Config — 모든 파라미터를 여기서 자유롭게 조절

`SqueezeV4Config`는 v3와 동일한 모든 필드 + SHAP 4개:
- `shap_caches`: 사용할 캐시 폴더 경로 tuple (`shap_cache/...`). 빈 tuple이면 v3 동작과 동일.
- `shap_top_k`: 캐시당 상위 K개 feature (0=전체)
- `shap_mode`: `'always_include'` (subset search 후보 X) | `'searchable'` (후보 O)
- `shap_prefix_with_tag`: 캐시 폴더명 prefix로 컬럼 충돌 방지 (기본 True)


In [3]:
cfg = SqueezeV4Config(
    # 베이스 풀 (hp variant만 - cell 8에서 filter)
    oof_rmse_cutoff=0.00560,
    no_clf=False,
    include_combined=False,
    include_ts_reg=False,

    # subset 크기 (hp variant만이라 풀이 작아짐 -> max도 축소)
    min_subset_size=3,
    max_subset_size=8,

    # SHAP X-stacking - 셀 11에서 zit_only/hp/002 mu+pi 2개로 갱신
    shap_caches=(),
    shap_top_k=20,
    shap_mode="always_include",
    shap_prefix_with_tag=True,

    # 탐색 (이전 v4 첫 run 수준 - 30~60분 목표)
    random_trials=500,
    local_seeds=6,
    local_steps=20,
    local_candidate_limit=30,
    top_refit=12,
    combo_refit=5,
    optuna_trials=50,

    # 선정 기준
    select_by="meta_cv_oof",
    val_gap_penalty=0.0,

    # 메타 규제
    enet_l1_ratio_grid=(0.05, 0.1, 0.2, 0.3, 0.5, 0.7),
    enet_alpha_n=20,
    ridge_alpha_grid=(1e-1, 3e-2, 1e-2, 3e-3, 1e-3, 1e-4),
    combo_seeds=(42, 123, 456, 789, 2024),

    cv_n_splits=5,

    # die->unit 집계
    agg_methods=("mean", "median", "max", "min",
                 "trimmed_mean", "weighted", "Q25", "Q75"),
    baseline_agg="mean",
    position_method="optuna",
    position_optuna_n_trials=30,
    use_pi_threshold=False,

    # iso 보정
    # 실험 A: base 위 iso 사전 적용(cell 14) + 메타 위 iso OFF
    # 실험 B: cell 5에서 use_iso=True로 바꾸고 처음부터 다시 실행 (이중 iso)
    use_iso=True,

    seed=None,
    deadline=None,
    deadline_margin_minutes=15.0,
    verbose=True,
)
print(f"seed={cfg.seed}")
print(f"project_root={cfg.project_root}")
print(f"result_dir={cfg.result_dir}")
print(f"shap_mode={cfg.shap_mode}, shap_top_k={cfg.shap_top_k}, n_caches={len(cfg.shap_caches)}")
print(f"use_iso={cfg.use_iso}  select_by={cfg.select_by}  val_gap_penalty={cfg.val_gap_penalty}")
print(f"탐색 budget: random={cfg.random_trials} local={cfg.local_seeds}x{cfg.local_steps} "
      f"optuna={cfg.optuna_trials} refit={cfg.top_refit} combo={cfg.combo_refit}")
assert cfg.output_dir.exists(), f"4_output 디렉토리가 없습니다: {cfg.output_dir}"

seed=3661380010
project_root=C:\Users\Dell5371\Desktop\기업연계프로젝트
result_dir=C:\Users\Dell5371\Desktop\기업연계프로젝트\4_output\04_stacking\results_extreme_v4
shap_mode=always_include, shap_top_k=20, n_caches=0
use_iso=True  select_by=meta_cv_oof  val_gap_penalty=0.0
탐색 budget: random=500 local=6x20 optuna=50 refit=12 combo=5


## 2. 모델 풀 탐색 (die-level CSV 보유 기준)


In [4]:
models = discovery.discover_models(cfg)
names = [m["name"] for m in models]
print(f"Filtered base models (die CSV 보유): {len(models)}")
for i, m in enumerate(models):
    print(f"{i:2d}  {m['oof_rmse']:.6f}  {m['category']:10s} {m['variant']:7s} {m['rel']}")

Filtered base models (die CSV 보유): 42
 0  0.005495  zit        pphp    01_zit\zit_only\pphp\001
 1  0.005495  zit        raw     01_zit\zit_only\raw\001
 2  0.005496  zit        hp      01_zit\zit_only\hp\002
 3  0.005497  reverse    hp      03_two_stage\reverse\hp\002
 4  0.005497  zit        hp      01_zit\zit_only\hp\001
 5  0.005497  reverse    hp      03_two_stage\reverse\hp\001
 6  0.005498  zit        pphp    01_zit\bag_zit\pphp\001
 7  0.005498  zit        hp      01_zit\bag_zit\hp\001
 8  0.005498  zit        raw     01_zit\bag_zit\raw\001
 9  0.005499  reverse    pphp    03_two_stage\reverse\pphp\001
10  0.005499  clf        hp      03_two_stage\default\clf\xgb\hp\001
11  0.005499  clf        hp      03_two_stage\default\clf\lgbm\hp\001
12  0.005500  clf        hp      03_two_stage\default\clf\lgbm\hp\002
13  0.005500  clf        hp      03_two_stage\default\clf\xgb\hp\002
14  0.005501  clf        pphp    03_two_stage\default\clf\lgbm\pphp\001
15  0.005502  clf        raw    

In [5]:
# ★ variant=hp만 유지 (사용자 결정: pphp, raw 제외 — hp가 가장 범용)
before = len(models)
models = [m for m in models if m["variant"] == "hp"]
names = [m["name"] for m in models]
print(f"hp filter: {before} → {len(models)} base 모델")
for i, m in enumerate(models):
    print(f"{i:2d}  {m['oof_rmse']:.6f}  {m['category']:10s} {m['rel']}")


hp filter: 42 → 24 base 모델
 0  0.005496  zit        01_zit\zit_only\hp\002
 1  0.005497  reverse    03_two_stage\reverse\hp\002
 2  0.005497  zit        01_zit\zit_only\hp\001
 3  0.005497  reverse    03_two_stage\reverse\hp\001
 4  0.005498  zit        01_zit\bag_zit\hp\001
 5  0.005499  clf        03_two_stage\default\clf\xgb\hp\001
 6  0.005499  clf        03_two_stage\default\clf\lgbm\hp\001
 7  0.005500  clf        03_two_stage\default\clf\lgbm\hp\002
 8  0.005500  clf        03_two_stage\default\clf\xgb\hp\002
 9  0.005503  clf        03_two_stage\default\clf\catboost\hp\001
10  0.005504  clf        03_two_stage\default\clf\catboost\hp\002
11  0.005507  clf        03_two_stage\default\clf\et\hp\002
12  0.005509  reg_single 02_reg_single\lgbm\hp\002
13  0.005510  zit        01_zit\zit_et\hp\002
14  0.005518  reg_single 02_reg_single\lgbm\hp\001
15  0.005519  reg_single 02_reg_single\xgb\hp\002
16  0.005519  reg_single 02_reg_single\et\hp\002
17  0.005522  reg_single 02_reg_single\

## 2.5 SHAP 캐시 확인 + cfg.shap_caches 설정

사용 가능한 캐시를 나열한 다음 cfg를 갱신해 원하는 캐시만 입력에 사용. **모든 캐시에 `run_wf_xy` 키가 있어야 한다** — 없으면 `build_shap_features.py`로 재생성 필요.


In [6]:
# 사용 가능한 SHAP 캐시 나열
import numpy as np
SHAP_CACHE_ROOT = Path(cfg.project_root) / "3_modeling" / "04_stacking" / "shap_cache"
available = []
for d in sorted(SHAP_CACHE_ROOT.iterdir() if SHAP_CACHE_ROOT.exists() else []):
    npz = d / "die_shap.npz"
    if not npz.exists():
        continue
    try:
        z = np.load(npz, allow_pickle=True)
        has_xy = "oof_run_wf_xy" in z.files
        n_feat = int(z["oof_shap"].shape[1]) if "oof_shap" in z.files else 0
    except Exception:
        has_xy = False
        n_feat = 0
    available.append({"name": d.name, "path": f"shap_cache/{d.name}",
                      "n_features": n_feat, "has_xy": has_xy})
df_cache = pd.DataFrame(available)
if df_cache.empty:
    print("⚠ SHAP 캐시가 없음 — build_shap_features.py / run_shap_all.py 먼저 실행")
else:
    n_total = len(df_cache)
    n_ok = int(df_cache['has_xy'].sum())
    print(f"총 캐시: {n_total}, run_wf_xy 키 있는 캐시: {n_ok}")
    if n_ok < n_total:
        print(f"⚠ {n_total - n_ok}개 캐시는 키 없음 → run_shap_all.py --force로 재생성 필요")
    display(df_cache)

총 캐시: 24, run_wf_xy 키 있는 캐시: 12
⚠ 12개 캐시는 키 없음 → run_shap_all.py --force로 재생성 필요


,name,path,n_features,has_xy
0,01_zit__bag_zit__hp__001__mu,shap_cache/01_zit__bag_zit__hp__001__mu,576,True
1,01_zit__zit_only__hp__001__mu,shap_cache/01_zit__zit_only__hp__001__mu,576,True
2,01_zit__zit_only__hp__001__pi,shap_cache/01_zit__zit_only__hp__001__pi,576,True
3,01_zit__zit_only__hp__002__mu,shap_cache/01_zit__zit_only__hp__002__mu,576,True
4,01_zit__zit_only__hp__002__pi,shap_cache/01_zit__zit_only__hp__002__pi,576,True
5,01_zit__zit_only__pphp__001__mu,shap_cache/01_zit__zit_only__pphp__001__mu,534,True
6,01_zit__zit_only__pphp__001__pi,shap_cache/01_zit__zit_only__pphp__001__pi,534,True
7,01_zit__zit_only__raw__001__mu,shap_cache/01_zit__zit_only__raw__001__mu,1034,True
8,01_zit__zit_only__raw__001__pi,shap_cache/01_zit__zit_only__raw__001__pi,1034,True
9,02_reg_single__catboost__hp__001,shap_cache/02_reg_single__catboost__hp__001,573,False


In [7]:
# SHAP 캐시 6개: zit_only/hp/002 mu+pi + clf hp/002 × 3 + bag_zit/hp/001 mu
# top_k=20으로 줄여서 캐시당 컬럼 20개. 총 SHAP = 120개 (base ~22와 합리적 비율).
from dataclasses import replace
cfg = replace(cfg, shap_caches=(
    'shap_cache/01_zit__zit_only__hp__002__mu',
    'shap_cache/01_zit__zit_only__hp__002__pi',
    'shap_cache/03_two_stage__default__clf__lgbm__hp__002',
    'shap_cache/03_two_stage__default__clf__xgb__hp__002',
    'shap_cache/03_two_stage__default__clf__catboost__hp__002',
    'shap_cache/01_zit__bag_zit__hp__001__mu',
))
print(f"적용된 shap_caches ({len(cfg.shap_caches)}개):")
for p in cfg.shap_caches:
    print(f"  - {p}")
print(f"mode={cfg.shap_mode}, top_k={cfg.shap_top_k}, prefix={cfg.shap_prefix_with_tag}")


적용된 shap_caches (6개):
  - shap_cache/01_zit__zit_only__hp__002__mu
  - shap_cache/01_zit__zit_only__hp__002__pi
  - shap_cache/03_two_stage__default__clf__lgbm__hp__002
  - shap_cache/03_two_stage__default__clf__xgb__hp__002
  - shap_cache/03_two_stage__default__clf__catboost__hp__002
  - shap_cache/01_zit__bag_zit__hp__001__mu
mode=always_include, top_k=20, prefix=True


## 3. ArrayBundle 한 줄 빌드

`search.build_array_bundle(cfg, models)`가 다음을 다 처리한다:
1. die-level base 행렬 oof/val/test 생성 + `(ufs_serial, run_wf_xy)` 정렬
2. cfg.shap_caches가 비어있지 않으면 SHAP die-level 컬럼 로드 + 정렬 + top-K 추출
3. shap_mode에 따라 X 결합 + names/extra_idx 구성
4. fast aggregator + unit y 배열 사전 캐싱


In [8]:
bundle = search.build_array_bundle(cfg, models)
print(f"X shape: oof={bundle.X_oof.shape}  val={bundle.X_val.shape}  test={bundle.X_test.shape}")
print(f"names: {len(bundle.names)} (base + searchable SHAP)")
print(f"extra_idx (always_include SHAP cols): {len(bundle.extra_idx)}개")
print(f"shap_mode={bundle.shap_mode}, extra_tags={bundle.extra_tags}")
print(f"unique units: oof={len(bundle.units_oof)}  val={len(bundle.units_val)}  test={len(bundle.units_test)}")
print(f"y_die_oof range: [{bundle.y_die_oof.min():.6f}, {bundle.y_die_oof.max():.6f}]")

  [shap cache] 01_zit__zit_only__hp__002__mu  (C:\Users\Dell5371\Desktop\기업연계프로젝트\3_modeling\04_stacking\shap_cache\01_zit__zit_only__hp__002__mu)
    [top-K] 20/576 컬럼
  [shap cache] 01_zit__zit_only__hp__002__pi  (C:\Users\Dell5371\Desktop\기업연계프로젝트\3_modeling\04_stacking\shap_cache\01_zit__zit_only__hp__002__pi)
    [top-K] 20/576 컬럼
  [shap cache] 03_two_stage__default__clf__lgbm__hp__002  (C:\Users\Dell5371\Desktop\기업연계프로젝트\3_modeling\04_stacking\shap_cache\03_two_stage__default__clf__lgbm__hp__002)
    [top-K] 20/576 컬럼
  [shap cache] 03_two_stage__default__clf__xgb__hp__002  (C:\Users\Dell5371\Desktop\기업연계프로젝트\3_modeling\04_stacking\shap_cache\03_two_stage__default__clf__xgb__hp__002)
    [top-K] 20/576 컬럼
  [shap cache] 03_two_stage__default__clf__catboost__hp__002  (C:\Users\Dell5371\Desktop\기업연계프로젝트\3_modeling\04_stacking\shap_cache\03_two_stage__default__clf__catboost__hp__002)
    [top-K] 20/576 컬럼
  [shap cache] 01_zit__bag_zit__hp__001__mu  (C:\Users\Dell5371\Desktop\기업연계프

In [9]:
# GroupKFold splits (oof에 대해서만 — meta_cv_oof 계산용)
cv_splits = discovery.make_group_kfold_splits(bundle.key_oof, n_splits=cfg.cv_n_splits, seed=cfg.seed)
print(f"CV: {len(cv_splits)}-fold GroupKFold by ufs_serial")
for i, (tr, vl) in enumerate(cv_splits):
    n_tr_unit = bundle.key_oof.iloc[tr]['ufs_serial'].nunique()
    n_vl_unit = bundle.key_oof.iloc[vl]['ufs_serial'].nunique()
    print(f"  fold {i}: tr_die={len(tr):>6d} ({n_tr_unit} units), vl_die={len(vl):>5d} ({n_vl_unit} units)")

CV: 5-fold GroupKFold by ufs_serial
  fold 0: tr_die= 83796 (20949 units), vl_die=20952 (5238 units)
  fold 1: tr_die= 83796 (20949 units), vl_die=20952 (5238 units)
  fold 2: tr_die= 83800 (20950 units), vl_die=20948 (5237 units)
  fold 3: tr_die= 83800 (20950 units), vl_die=20948 (5237 units)
  fold 4: tr_die= 83800 (20950 units), vl_die=20948 (5237 units)


## 3.5 SHAP cache ablation 실험 (어떤 캐시 조합이 val을 가장 낮추는가)

각 config: random 200 trials만, local/refit/optuna 생략 → 빠르게 fast 평가. 11 config × ~6분 ≈ 70분.

**`top_oof_val` 기준으로 정렬** — 실제 운영에선 oof top 1을 픽하므로 그 val이 진짜 비교 지표. `best_val`만 보면 우연히 val 낮은 노이즈 record가 잡힘.

결과 보고 우승 config을 cell 7의 `cfg.shap_caches`에 박은 뒤 셀 8(bundle)부터 다시 실행.

In [10]:
# === SHAP cache ablation: 어떤 캐시 조합이 val을 가장 낮추는가? ===
from dataclasses import replace
import time
import pandas as pd

ALL_CACHES = (
    'shap_cache/01_zit__zit_only__hp__002__mu',
    'shap_cache/01_zit__zit_only__hp__002__pi',
    'shap_cache/03_two_stage__default__clf__lgbm__hp__002',
    'shap_cache/03_two_stage__default__clf__xgb__hp__002',
    'shap_cache/03_two_stage__default__clf__catboost__hp__002',
    'shap_cache/01_zit__bag_zit__hp__001__mu',
)
NAMES = ['zit_mu', 'zit_pi', 'clf_lgbm', 'clf_xgb', 'clf_cat', 'bag_mu']

CONFIGS = [
    ('baseline_no_shap', ()),
    *[(f'only_{NAMES[i]}', (ALL_CACHES[i],)) for i in range(6)],
    ('mu_only',   (ALL_CACHES[0], ALL_CACHES[5])),
    ('pi_only',   (ALL_CACHES[1],)),
    ('clf_3',     (ALL_CACHES[2], ALL_CACHES[3], ALL_CACHES[4])),
    ('mu+pi',     (ALL_CACHES[0], ALL_CACHES[1], ALL_CACHES[5])),
    ('all_6',     ALL_CACHES),
]

# 빠른 비교용 cfg — local/refit/optuna 끄고 random만, top_k도 10으로 줄여 ratio 부담↓
cfg_ablate = replace(cfg,
    random_trials=200, local_seeds=0, local_steps=0,
    optuna_trials=0, top_refit=0, combo_refit=0,
    shap_top_k=10,
    verbose=False,
)

rows = []
for cfg_name, caches in CONFIGS:
    t0 = time.time()
    cfg_i = replace(cfg_ablate, shap_caches=caches)
    try:
        b = search.build_array_bundle(cfg_i, models)
        recs, _ = search.run_search_stages(models, b, cfg_i)
        best_oof = min(r.oof_rmse for r in recs)
        best_val = min(r.val_rmse for r in recs)
        top_by_oof = min(recs, key=lambda r: r.oof_rmse)
        rows.append({
            'config': cfg_name,
            'n_caches': len(caches),
            'n_shap_cols': len(b.extra_idx),
            'best_oof': best_oof,
            'best_val': best_val,
            'top_oof_val': top_by_oof.val_rmse,
            'gap': top_by_oof.val_rmse - top_by_oof.oof_rmse,
            'n_trials': len(recs),
            'elapsed_s': round(time.time()-t0, 1),
        })
        print(f"  {cfg_name:20s} | oof={best_oof:.6f} val={best_val:.6f} "
              f"top_oof_val={top_by_oof.val_rmse:.6f} | {rows[-1]['elapsed_s']}s")
    except Exception as e:
        print(f"  {cfg_name:20s} | FAILED: {e}")

ablate_df = pd.DataFrame(rows).sort_values('top_oof_val').reset_index(drop=True)
print('\n=== ablation (top_oof_val 기준 ranking) ===')
print(ablate_df.to_string(index=False))

  baseline_no_shap     | oof=0.005487 val=0.005700 top_oof_val=0.005703 | 55.3s
  only_zit_mu          | oof=0.005486 val=0.005700 top_oof_val=0.005702 | 162.6s
  only_zit_pi          | oof=0.005486 val=0.005700 top_oof_val=0.005703 | 163.4s
  only_clf_lgbm        | oof=0.005486 val=0.005700 top_oof_val=0.005702 | 163.2s
  only_clf_xgb         | oof=0.005486 val=0.005700 top_oof_val=0.005703 | 172.2s
  only_clf_cat         | oof=0.005486 val=0.005700 top_oof_val=0.005703 | 166.9s
  only_bag_mu          | oof=0.005487 val=0.005699 top_oof_val=0.005702 | 163.4s
  mu_only              | oof=0.005485 val=0.005700 top_oof_val=0.005702 | 227.2s
  pi_only              | oof=0.005486 val=0.005700 top_oof_val=0.005703 | 163.8s
  clf_3                | oof=0.005485 val=0.005701 top_oof_val=0.005703 | 339.8s
  mu+pi                | oof=0.005484 val=0.005700 top_oof_val=0.005703 | 321.0s
  all_6                | oof=0.005482 val=0.005701 top_oof_val=0.005704 | 573.9s

=== ablation (top_oof_val 기준

## 4. Fast 탐색 (seed → random → local → optuna)

탐색 단계는 빠르게 unit RMSE 추정을 위해 **단순 mean 집계** 사용 (refit에서 정밀 집계로 재평가). SHAP `always_include` 모드면 subset search 후보는 base만, 메타 학습 시 SHAP 자동 추가.


In [11]:
records, rng = search.run_search_stages(models, bundle, cfg)
print(f"\nTotal fast re2cords: {len(records)}")


[seed subsets] 7

[random search] 500 subsets
    500/500  t= 2968.3s  best_oof=0.005473711  val_best=0.005701562

[local search] seeds=6
  [candidate_pool] size=24
  local  1: best_oof=0.005472733 val=0.005703610 k=8


KeyboardInterrupt: 

## 5. Refit (정밀 메타: ENet / ENetPositive / Combo + tune_and_apply)


In [ ]:
refit_records, artifacts = search.run_refit_stage(
    records, bundle, cfg, cv_splits=cv_splits,
)
all_records = records + refit_records
print(f"\nTotal records (fast + refit): {len(all_records)}")


[refit] top_refit=15, combo_refit=5, unique subsets=15


c:\Users\Dell5371\anaconda3\Lib\site-packages\optuna\_experimental.py:33: ExperimentalWarning: Argument ``multivariate`` is an experimental feature. The interface can change in the future.
  optuna_warn(
c:\Users\Dell5371\anaconda3\Lib\site-packages\optuna\_experimental.py:33: ExperimentalWarning: Argument ``group`` is an experimental feature. The interface can change in the future.
  optuna_warn(


[Position weights / Optuna 30t] best=0.005486, w=[0.5, 0.311, 0.148, 0.041]
[Aggregation] RMSEs: {'mean': 0.005487, 'median': 0.005487, 'max': 0.005493, 'min': 0.005491, 'trimmed_mean': 0.005487, 'weighted': 0.005486, 'Q25': 0.005487, 'Q75': 0.005489}
[Aggregation] best=weighted (0.005486)
[zero_clip] best=0.0010 (0.005490)
[Postprocess] best_agg=mean, pi_th=None, zero_clip=None, train_rmse=0.005487, val_rmse=0.005703
  baseline_mean                  val_rmse=0.0057025031604990865
  after_agg(mean)                val_rmse=0.0057025031604990865
  after_pi_th                    val_rmse=0.0057025031604990865
  after_zero_clip                val_rmse=0.0057025031604990865
  [decision] aggregation    weighted rejected (val 0.005703 <= 0.005703) -> keep mean
  [decision] zero_clip      0.0010 rejected (val 0.005703 <= 0.005704) — skip


c:\Users\Dell5371\anaconda3\Lib\site-packages\optuna\_experimental.py:33: ExperimentalWarning: Argument ``multivariate`` is an experimental feature. The interface can change in the future.
  optuna_warn(
c:\Users\Dell5371\anaconda3\Lib\site-packages\optuna\_experimental.py:33: ExperimentalWarning: Argument ``group`` is an experimental feature. The interface can change in the future.
  optuna_warn(


[Position weights / Optuna 30t] best=0.005491, w=[0.323, 0.386, 0.257, 0.034]
[Aggregation] RMSEs: {'mean': 0.005491, 'median': 0.005491, 'max': 0.005496, 'min': 0.005495, 'trimmed_mean': 0.005491, 'weighted': 0.005491, 'Q25': 0.005492, 'Q75': 0.005493}
[Aggregation] best=weighted (0.005491)
[zero_clip] best=0.0010 (0.005494)
[Postprocess] best_agg=weighted, pi_th=None, zero_clip=None, train_rmse=0.005491, val_rmse=0.005702
  baseline_mean                  val_rmse=0.005702820254789871
  after_agg(weighted)            val_rmse=0.005702381312735033
  after_pi_th                    val_rmse=0.005702381312735033
  after_zero_clip                val_rmse=0.005702381312735033
  [decision] aggregation    weighted adopted (val 0.005703 -> 0.005702)
  [decision] zero_clip      0.0010 rejected (val 0.005702 <= 0.005703) — skip


...

[Position weights / Optuna 30t] best=0.005491, w=[0.323, 0.386, 0.257, 0.034]
[Aggregation] RMSEs: {'mean': 0.005491, 'median': 0.005491, 'max': 0.005496, 'min': 0.005494, 'trimmed_mean': 0.005491, 'weighted': 0.005491, 'Q25': 0.005492, 'Q75': 0.005493}
[Aggregation] best=weighted (0.005491)
[zero_clip] best=0.0010 (0.005494)
[Postprocess] best_agg=weighted, pi_th=None, zero_clip=0.001, train_rmse=0.005494, val_rmse=0.005703
  baseline_mean                  val_rmse=0.00570314830870367
  after_agg(weighted)            val_rmse=0.005702725136092433
  after_pi_th                    val_rmse=0.005702725136092433
  after_zero_clip                val_rmse=0.0057026532369250415
  [decision] aggregation    weighted adopted (val 0.005703 -> 0.005703)
  [decision] zero_clip      0.0010 adopted (val 0.005703 -> 0.005703)
  refit  14/15  best_meta_cv_oof=0.005488322  mcv_unit[ENet=0.005489, ENetPositive=0.005492]  val_best=0.005700866


c:\Users\Dell5371\anaconda3\Lib\site-packages\optuna\_experimental.py:33: ExperimentalWarning: Argument ``multivariate`` is an experimental feature. The interface can change in the future.
  optuna_warn(
c:\Users\Dell5371\anaconda3\Lib\site-packages\optuna\_experimental.py:33: ExperimentalWarning: Argument ``group`` is an experimental feature. The interface can change in the future.
  optuna_warn(


[Position weights / Optuna 30t] best=0.005487, w=[0.39, 0.364, 0.191, 0.055]
[Aggregation] RMSEs: {'mean': 0.005488, 'median': 0.005488, 'max': 0.005494, 'min': 0.005491, 'trimmed_mean': 0.005488, 'weighted': 0.005487, 'Q25': 0.005488, 'Q75': 0.00549}
[Aggregation] best=weighted (0.005487)
[zero_clip] best=0.0010 (0.005490)
[Postprocess] best_agg=mean, pi_th=None, zero_clip=None, train_rmse=0.005488, val_rmse=0.005702
  baseline_mean                  val_rmse=0.005702399390346765
  after_agg(mean)                val_rmse=0.005702399390346765
  after_pi_th                    val_rmse=0.005702399390346765
  after_zero_clip                val_rmse=0.005702399390346765
  [decision] aggregation    weighted rejected (val 0.005702 <= 0.005702) -> keep mean
  [decision] zero_clip      0.0010 rejected (val 0.005702 <= 0.005703) — skip


c:\Users\Dell5371\anaconda3\Lib\site-packages\optuna\_experimental.py:33: ExperimentalWarning: Argument ``multivariate`` is an experimental feature. The interface can change in the future.
  optuna_warn(
c:\Users\Dell5371\anaconda3\Lib\site-packages\optuna\_experimental.py:33: ExperimentalWarning: Argument ``group`` is an experimental feature. The interface can change in the future.
  optuna_warn(


[Position weights / Optuna 30t] best=0.005491, w=[0.323, 0.386, 0.257, 0.034]
[Aggregation] RMSEs: {'mean': 0.005491, 'median': 0.005491, 'max': 0.005496, 'min': 0.005495, 'trimmed_mean': 0.005491, 'weighted': 0.005491, 'Q25': 0.005492, 'Q75': 0.005493}
[Aggregation] best=weighted (0.005491)
[zero_clip] best=0.0010 (0.005494)
[Postprocess] best_agg=weighted, pi_th=None, zero_clip=None, train_rmse=0.005491, val_rmse=0.005702
  baseline_mean                  val_rmse=0.005702913216021593
  after_agg(weighted)            val_rmse=0.005702487565472556
  after_pi_th                    val_rmse=0.005702487565472556
  after_zero_clip                val_rmse=0.005702487565472556
  [decision] aggregation    weighted adopted (val 0.005703 -> 0.005702)
  [decision] zero_clip      0.0010 rejected (val 0.005702 <= 0.005703) — skip
  refit  15/15  best_meta_cv_oof=0.005488322  mcv_unit[ENet=0.005489, ENetPositive=0.005492]  val_best=0.005700866

Total records (fast + refit): 2177


## 6. 저장

- `summary.json` — cfg + records + **shap 메타** (v4 신규)
- `summary.csv` — 전체 records (score 정렬, `n_extra` 컬럼 포함)
- `best_die_oof.csv / val / test` — 1등 die-level pred (iso 적용 후)
- `best_unit_oof.csv / val / test` — 1등 unit-level pred (집계 후 최종)
- `best_weights.json` — 1등 record의 메타 가중치 (base + SHAP 컬럼 모두 박제)


In [ ]:
run_dir = io.save_outputs(
    all_records, artifacts, bundle, cfg, models,
    top_n_weights=1,
)
print(f"\nrun_dir = {run_dir}")


FINAL TOP 20 BY META_CV_OOF OBJECTIVE
 1. obj=0.005488322 oof=0.005487 val=0.005703 test=0.008406 mcv_unit=0.005488 agg=mean          k=12 refit__ENet__k12
 2. obj=0.005488337 oof=0.005487 val=0.005703 test=0.008406 mcv_unit=0.005488 agg=mean          k=12 refit__ENet__k12
 3. obj=0.005488396 oof=0.005487 val=0.005703 test=0.008406 mcv_unit=0.005488 agg=mean          k=12 refit__ENet__k12
 4. obj=0.005488419 oof=0.005487 val=0.005703 test=0.008407 mcv_unit=0.005488 agg=mean          k=11 refit__ENet__k11
 5. obj=0.005488517 oof=0.005487 val=0.005702 test=0.008406 mcv_unit=0.005489 agg=mean          k=12 refit__ENet__k12
 6. obj=0.005488531 oof=0.005487 val=0.005702 test=0.008406 mcv_unit=0.005489 agg=mean          k=11 refit__ENet__k11
 7. obj=0.005488533 oof=0.005487 val=0.005703 test=0.008407 mcv_unit=0.005489 agg=mean          k=12 refit__ENet__k12
 8. obj=0.005488603 oof=0.005487 val=0.005702 test=0.008406 mcv_unit=0.005489 agg=mean          k=12 refit__ENet__k12
 9. obj=0.0054886

## 7. (옵션) 상위 record 비교 / 가중치 들여다보기


In [ ]:
import json
df_sum = pd.read_csv(run_dir / "summary.csv").head(20)
df_sum

,tag,stage,method,n_base,val_rmse,test_rmse,oof_rmse,val_rmse_die,test_rmse_die,oof_rmse_die,meta_cv_oof_rmse,aggregation,pi_threshold,zero_clip,n_extra,extra_tags,objective,objective_score
0,refit__ENet__k12,refit,ENet,12,0.005703,0.008406,0.005487,0.005704,0.008407,0.005490,0.005488,mean,NaN,NaN,0,[],meta_cv_oof,0.005488
1,refit__ENet__k12,refit,ENet,12,0.005703,0.008406,0.005487,0.005704,0.008408,0.005490,0.005488,mean,NaN,NaN,0,[],meta_cv_oof,0.005488
2,refit__ENet__k12,refit,ENet,12,0.005703,0.008406,0.005487,0.005704,0.008408,0.005490,0.005488,mean,NaN,NaN,0,[],meta_cv_oof,0.005488
3,refit__ENet__k11,refit,ENet,11,0.005703,0.008407,0.005487,0.005705,0.008408,0.005490,0.005488,mean,NaN,NaN,0,[],meta_cv_oof,0.005488
4,refit__ENet__k12,refit,ENet,12,0.005702,0.008406,0.005487,0.005704,0.008407,0.005490,0.005489,mean,NaN,NaN,0,[],meta_cv_oof,0.005489
5,refit__ENet__k11,refit,ENet,11,0.005702,0.008406,0.005487,0.005704,0.008407,0.005490,0.005489,mean,NaN,NaN,0,[],meta_cv_oof,0.005489
6,refit__ENet__k12,refit,ENet,12,0.005703,0.008407,0.005487,0.005705,0.008408,0.005490,0.005489,mean,NaN,NaN,0,[],meta_cv_oof,0.005489
7,refit__ENet__k12,refit,ENet,12,0.005702,0.008406,0.005487,0.005704,0.008407,0.005490,0.005489,mean,NaN,NaN,0,[],meta_cv_oof,0.005489
8,refit__ENet__k12,refit,ENet,12,0.005703,0.008407,0.005487,0.005705,0.008408,0.005490,0.005489,mean,NaN,NaN,0,[],meta_cv_oof,0.005489
9,refit__ENet__k11,refit,ENet,11,0.005702,0.008407,0.005486,0.005704,0.008408,0.005490,0.005489,weighted,NaN,NaN,0,[],meta_cv_oof,0.005489


In [ ]:
import json
with open(run_dir / "best_weights.json", "r", encoding="utf-8") as f:
    bw = json.load(f)
best = bw['weights'][0]
print(f"tag={best['tag']}  method={best['method']}  shap_mode={best.get('shap_mode')}")
print(f"n_base={best['n_base']}, n_extra={best['n_extra']}")
print(f"extra_tags={best.get('extra_tags')}")
print(f"\nbase_models ({len(best['base_models'])}개):")
for n in best['base_models']:
    print(f"  - {n}")
if best.get('extra_features'):
    print(f"\nextra_features ({len(best['extra_features'])}개, 처음 10개만):")
    for n in best['extra_features'][:10]:
        print(f"  - {n}")
# 가중치 top-10 (절대값 기준)
learner = best.get('learner', {})
if 'weights' in learner:
    ws = sorted(learner['weights'], key=lambda x: -abs(x.get('w', 0)))[:10]
    print(f"\n가중치 top-10 (|w| 기준):")
    for w in ws:
        print(f"  {w['name']:<60s}  w={w['w']:+.6f}")


tag=refit__ENet__k12  method=ENet  shap_mode=none
n_base=12, n_extra=0
extra_tags=[]

base_models (12개):
  - 01_zit__zit_only__pphp__001
  - 01_zit__zit_only__raw__001
  - 03_two_stage__reverse__hp__002
  - 03_two_stage__reverse__pphp__001
  - 03_two_stage__default__clf__xgb__hp__001
  - 03_two_stage__default__clf__xgb__hp__002
  - 03_two_stage__default__clf__lgbm__pphp__001
  - 02_reg_single__lgbm__hp__002
  - 02_reg_single__xgb__hp__002
  - 02_reg_single__et__hp__002
  - 02_reg_single__lgbm__pphp__001
  - 02_reg_single__et__hp__001

가중치 top-10 (|w| 기준):
  03_two_stage__default__clf__xgb__hp__002                      w=-0.000544
  01_zit__zit_only__raw__001                                    w=+0.000473
  03_two_stage__default__clf__xgb__hp__001                      w=+0.000440
  01_zit__zit_only__pphp__001                                   w=+0.000418
  03_two_stage__reverse__hp__002                                w=+0.000398
  02_reg_single__et__hp__002                              

## 8. 2-stage selection — 'oof best + val 낮은 풀' 찾기

`val_gap_penalty=0` 유지하므로 selection 자체엔 val 정보를 안 쓰지만, oof top 후보들 중 val이 가장 낮은 record를 picking하는 추가 단계.


In [ ]:
# 2-stage selection — refit이 있으면 mcv 기준, 없으면 fast oof 기준으로 fallback
TOP_K_OOF = 30

df = pd.read_csv(run_dir / "summary.csv")
refit_df = df[df["stage"] == "refit"].copy()

if len(refit_df) == 0:
    # refit이 안 돌았다 → deadline이 너무 가까웠거나 cell 17이 안 돌았음
    print(f"⚠ refit records가 0개. fast records의 oof_rmse 기준으로 fallback.")
    print(f"   (mcv_unit은 refit 단계에서만 계산되므로 unavailable)")
    fallback_pool = df.copy()
    sort_col = "oof_rmse"
    pool_label = "fast pool"
else:
    fallback_pool = refit_df
    sort_col = "meta_cv_oof_rmse"
    pool_label = "refit pool (mcv_unit)"

# 1차: 정렬 기준 컬럼으로 상위 K — "OOF 잘 나오는 후보"
top_oof = fallback_pool.nsmallest(TOP_K_OOF, sort_col)
if len(top_oof) == 0:
    raise RuntimeError(
        f"summary.csv에 정렬 가능한 record가 0개. run_dir={run_dir}\n"
        f"  total records={len(df)}, refit={len(refit_df)}, sort_col={sort_col}"
    )

# 2차: 그 안에서 val_rmse 낮은 순
top_oof_sorted_by_val = top_oof.sort_values("val_rmse").reset_index(drop=True)
print(f"\n=== {pool_label} TOP {min(TOP_K_OOF, len(top_oof))} 중 val_rmse 낮은 순 TOP 10 ===")
print(top_oof_sorted_by_val[
    ["tag", "method", "n_base", sort_col, "oof_rmse", "val_rmse", "test_rmse", "aggregation"]
].head(10).to_string(index=False))

# 동질성 진단
v_min, v_max = top_oof[sort_col].min(), top_oof[sort_col].max()
print(f"\n{sort_col} range in TOP {len(top_oof)}: [{v_min:.6f}, {v_max:.6f}]  span={v_max - v_min:.6f}")
print("  → span이 0.0001 미만이면 TOP K 내부는 거의 동급 (val peek 안전).")

# 1등 record 상세
best_pick = top_oof_sorted_by_val.iloc[0]
print(f"\n--- 1st pick (2-stage selection) ---")
print(f"  tag       : {best_pick['tag']}")
print(f"  method    : {best_pick['method']}")
print(f"  n_base    : {int(best_pick['n_base'])}")
print(f"  {sort_col:9s}: {best_pick[sort_col]:.6f}")
print(f"  oof       : {best_pick['oof_rmse']:.6f}  (in-sample 진단치)")
print(f"  val       : {best_pick['val_rmse']:.6f}  ← 2-stage selection의 정렬 키")
print(f"  test      : {best_pick['test_rmse']:.6f}")
print(f"  agg       : {best_pick['aggregation']}")


=== refit pool (mcv_unit) TOP 30 중 val_rmse 낮은 순 TOP 10 ===
                     tag       method  n_base  meta_cv_oof_rmse  oof_rmse  val_rmse  test_rmse aggregation
        refit__ENet__k12         ENet      12          0.005489  0.005487  0.005702   0.008405        mean
        refit__ENet__k11         ENet      11          0.005489  0.005487  0.005702   0.008406        mean
        refit__ENet__k12         ENet      12          0.005489  0.005487  0.005702   0.008406        mean
        refit__ENet__k12         ENet      12          0.005489  0.005487  0.005702   0.008406        mean
refit__ENetPositive__k11 ENetPositive      11          0.005492  0.005491  0.005702   0.008406    weighted
refit__ENetPositive__k12 ENetPositive      12          0.005492  0.005491  0.005702   0.008406    weighted
refit__ENetPositive__k12 ENetPositive      12          0.005492  0.005491  0.005702   0.008406    weighted
        refit__ENet__k11         ENet      11          0.005489  0.005486  0.005702

---
> **참고**: v3 노트북의 마지막 '핵심 X 컬럼 리스트' 셀(`EXTRA_FOLDERS`)은 v4에서 제거됨. SHAP X-stacking이 그 역할을 대체.
